# Data Preparation (Polutan CO - Kecamatan Bungah)

Notebook ini mengeksekusi tahapan **Data Preparation** metodologi CRISP-DM untuk polutan Karbon Monoksida ($	ext{CO}$) pada skala geografis **Kecamatan Bungah, Kabupaten Gresik** selama 365 hari (24 Agustus 2025 – 23 Agustus 2026):

**Alur Pemrosesan Data:**
1. **Penanganan Missing Values & Outliers** (Deteksi Outlier $\rightarrow$ Pengosongan Outlier $\rightarrow$ Linear Time Interpolation Sekaligus)
2. **Penyimpanan Dataset Clean** (`CO_clean.csv`)
3. **Ekstraksi 68 Fitur Time-Series TSFEL** ($f_1 \dots f_{68}$)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tsfel
import os

# Set style visualisasi plot
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Penanganan Missing Values & Outliers

### 1.1 Formula & Mekanisme Interpolasi Linear

Interpolasi linear mengestimasi nilai sel kosong $X(t)$ pada tanggal $t$ yang berada di antara dua titik observasi valid terdekat $X(t_1)$ dan $X(t_2)$ dengan $t_1 < t < t_2$:

$$X(t) = X(t_1) + \left( \frac{t - t_1}{t_2 - t_1} \right) \cdot \left[ X(t_2) - X(t_1) \right]$$

### 1.2 Alur Pemrosesan Berurutan:
1. Menghitung jumlah data kosong (`NaN`) awal pada data mentah $\text{CO}$.
2. Mengidentifikasi data pencilan (*outliers*) menggunakan metode Interquartile Range (IQR):
   $$\text{IQR} = Q_3 - Q_1$$
   $$\text{Batas Bawah} = Q_1 - 1.5 \times \text{IQR}, \quad \text{Batas Atas} = Q_3 + 1.5 \times \text{IQR}$$
3. Mengosongkan nilai tanggal *outlier* tersebut menjadi `NaN`.
4. Melakukan **Linear Time Interpolation** secara bersamaan untuk seluruh titik `NaN` agar diperoleh sinyal kontinu mulus 365 hari tanpa pencilan ekstrem.

In [2]:
# 1. Membaca Dataset Mentah CO (365 Hari)
raw_path = '../data/csv/CO_gresik_timeseries.csv'
df = pd.read_csv(raw_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

nan_awal = df['CO'].isna().sum()
print(f"Jumlah NaN Awal                    : {nan_awal} hari ({nan_awal/len(df)*100:.2f}%)")

# 2. Deteksi Outlier pada Data Valid Menggunakan Metode IQR
q1 = df['CO'].quantile(0.25)
q3 = df['CO'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

is_outlier = (df['CO'] < lower_bound) | (df['CO'] > upper_bound)
jumlah_outlier = is_outlier.sum()
print(f"Kuartil 1 (Q1)                     : {q1:.6f}")
print(f"Kuartil 3 (Q3)                     : {q3:.6f}")
print(f"IQR (Q3 - Q1)                      : {iqr:.6f}")
print(f"Batas Bawah (Lower)                : {lower_bound:.6f}")
print(f"Batas Atas (Upper)                 : {upper_bound:.6f}")
print(f"Outlier Terdeteksi                 : {jumlah_outlier} hari")

# 3. Mengosongkan Nilai Outlier Menjadi NaN (Sesuai Strategi Data Cleaning)
df.loc[is_outlier, 'CO'] = np.nan
nan_setelah_outlier = df['CO'].isna().sum()
print(f"Jumlah NaN Setelah Outlier Dikosongkan: {nan_setelah_outlier} hari")

# 4. Imputasi Linear Time Interpolation Sekaligus!
df['CO_clean'] = df['CO'].interpolate(method='linear', limit_direction='both')
nan_akhir = df['CO_clean'].isna().sum()
print(f"Jumlah NaN Setelah Imputasi Akhir   : {nan_akhir} hari (100% Clean!)")

Jumlah NaN Awal                    : 173 hari (47.40%)
Kuartil 1 (Q1)                     : 0.026530
Kuartil 3 (Q3)                     : 0.030774
IQR (Q3 - Q1)                      : 0.004244
Batas Bawah (Lower)                : 0.020164
Batas Atas (Upper)                 : 0.037140
Outlier Terdeteksi                 : 11 hari
Jumlah NaN Setelah Outlier Dikosongkan: 184 hari
Jumlah NaN Setelah Imputasi Akhir   : 0 hari (100% Clean!)


## 2. Menyimpan Dataset Hasil Pembersihan (`CO_clean.csv`)

Data yang telah bersih dari *missing values* dan *outliers* disimpan ke dalam file CSV baru `data/csv/CO_clean.csv` untuk menjaga keaslian data mentah.

In [3]:
# Simpan Data Super Clean ke File CSV yang Simpel & Mudah Diingat
df_clean = df[['date', 'CO_clean']].rename(columns={'CO_clean': 'CO'})
clean_path = '../data/csv/CO_clean.csv'
df_clean.to_csv(clean_path, index=False)

print(f"SUCCESS: Dataset bersih disimpan ke '{clean_path}' ({len(df_clean)} baris data)")

SUCCESS: Dataset bersih disimpan ke '../data/csv/CO_clean.csv' (365 baris data)


## 3. Ekstraksi 68 Fitur Time-Series (TSFEL Library)

Pustaka **TSFEL (Time Series Feature Extraction Library)** digunakan untuk merepresentasikan sinyal $\text{CO}$ selama 365 hari ke dalam **68 fitur perwakilan** ($f_1$ hingga $f_{68}$) yang terbagi ke dalam 3 domain utama:
1. **Domain Statistik ($f_1 \dots f_{20}$)**: Mengukur karakteristik pemusatan, sebaran, kemiringan, dan distribusi probabilitas sinyal.
2. **Domain Temporal ($f_{21} \dots f_{41}$)**: Mengukur sifat linier, autokorelasi, frekuensi perlintasan nol, dan durasi fluktuasi dalam domain waktu.
3. **Domain Spektral ($f_{42} \dots f_{68}$)**: Mengukur distribusi energi frekuensi sinyal menggunakan transformasi Fourier (*Fast Fourier Transform* / FFT) dan Wavelet CWT.

In [4]:
# 1. Memuat Katalog Konfigurasi Fitur TSFEL
cfg = tsfel.get_features_by_domain()

# Penyesuaian max_width=1 pada 4 fitur wavelet untuk menghasilkan 68 fitur unik yang terurut rapi
wavelet_feats = ['Wavelet absolute mean', 'Wavelet energy', 'Wavelet standard deviation', 'Wavelet variance']
for feat in wavelet_feats:
    if feat in cfg.get('spectral', {}):
        cfg['spectral'][feat]['parameters']['max_width'] = 1

print("Katalog fitur TSFEL berhasil dimuat dan disesuaikan!")

Katalog fitur TSFEL berhasil dimuat dan disesuaikan!


In [5]:
# 2. Mengekstrak 68 Fitur dari Sinyal CO Clean (365 Hari)
features_df = tsfel.time_series_features_extractor(cfg, df_clean['CO'], fs=1, verbose=0)

# 3. Format Tabel f1 hingga f68 Sesuai Penomoran Tugas
tabel_fitur = features_df.T.reset_index()
tabel_fitur.columns = ['Nama Fitur TSFEL', 'Nilai Fitur']
tabel_fitur.index = [f'f{i+1}' for i in range(len(tabel_fitur))]

# 4. Simpan Matriks Fitur ke File CSV
features_path = '../data/csv/CO_tsfel_features.csv'
features_df.to_csv(features_path, index=False)
print(f"SUCCESS: {len(tabel_fitur)} Fitur TSFEL berhasil diekstrak dan disimpan ke '{features_path}'!")

C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\tsfel\feature_extraction\calc_features.py:195: SyntaxWarning: invalid escape sequence '\*'
  \**kwargs:


IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
# 5. Tampilkan Tabel Ringkasan Fitur (f1 hingga f68)
tabel_fitur